# Week 03 | Reading and checking short Python programs

**Core practical: 45 minutes.** Diagnose a silent denominator error using small tests.

No paid AI tool, local installation or external dataset download is required. In Colab, upload this notebook through File > Upload notebook, then run cells from top to bottom. Local Jupyter with Python 3.9+ is an alternative. Plotting is optional.

**Data boundary:** Every generated value is synthetic. Explicit REAL EXCERPT / REAL record cards are separately labelled and limited to their stated purpose. No patient data should be entered.

## Before running (5 min)
GC fraction counts G and C bases among the bases your denominator actually includes.

Write a prediction in the response cell before executing the analysis.

## Setup (5 min)
Replace only `COURSE_ID` with your assigned pseudonym. A seed supports reproducibility; it is not proof of authorship.

In [ ]:
import hashlib, json, math, random, statistics, sys
from pathlib import Path
COURSE_ID = "demo-001"  # Replace with your assigned course pseudonym, not your name or national ID.
SEED = int(hashlib.sha256(COURSE_ID.encode()).hexdigest()[:8], 16)
rng = random.Random(SEED)
RESULTS = {}
print("Python", sys.version.split()[0], "| course ID", COURSE_ID, "| seed", SEED)

def mean(values):
    if not values: raise ValueError("Cannot average an empty list")
    return sum(values) / len(values)

def bh_adjust(pvalues):
    """Benjamini-Hochberg adjusted p-values, returned in original order."""
    if any(not 0 <= p <= 1 for p in pvalues): raise ValueError("p must be in [0,1]")
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    out = [0.0] * m
    running = 1.0
    for j in range(m - 1, -1, -1):
        i = order[j]
        running = min(running, pvalues[i] * m / (j + 1))
        out[i] = running
    return out

def optional_plot(labels, values, ylabel, title):
    """Plot when matplotlib is available; numerical work never requires it."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Plot unavailable; use the numerical table above.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    ax.set(ylabel=ylabel, title=title)
    fig.tight_layout()
    plt.show()

### Prediction
Write your prediction here before running the investigation, then copy it into PREDICTION in the response cell.

## Guided investigation (20 min)
1. Trace the defective function by hand on GCNN.
2. Read the corrected function and explain each change.
3. Add one new input-output test.
4. State the biological meaning of the returned value.

In [ ]:
def gc_bug(seq):
    # Intentional defect: silently includes N in denominator and fails on empty input.
    return sum(b in "GC" for b in seq.upper()) / len(seq)

def gc_fixed(seq):
    seq = seq.upper()
    if any(b not in "ACGTN" for b in seq): raise ValueError("Unsupported symbol")
    known = [b for b in seq if b in "ACGT"]
    return sum(b in "GC" for b in known) / len(known) if known else None

for text in ["ACGTGC", "GCNN", "NN", ""]:
    try: before = gc_bug(text)
    except ZeroDivisionError: before = "ERROR: division by zero"
    print(repr(text), "buggy:", before, "fixed:", gc_fixed(text))
assert gc_fixed("atat") == 0.0
assert gc_fixed("GCGC") == 1.0
assert gc_fixed("") is None
# Add your own small test here; reading and testing are core, writing a new function is optional.
assert gc_fixed("AN") == 0.0
RESULTS = {"data_status": "SYNTHETIC", "GCNN_bug": gc_bug("GCNN"), "GCNN_fixed": gc_fixed("GCNN")}

## Explain the evidence (10 min)
**Q1.** Why can a program run without an error and still be scientifically wrong?

**Q2.** Explain the difference between returning None and returning zero for NN.

**Q3.** Give your own test with input, expected output and the assumption being checked.

In [ ]:
PREDICTION = ""  # Fill before the analysis.
RESPONSES = {"Q1": "", "Q2": "", "Q3": ""}
AI_DISCLOSURE = "No AI used."  # Change to tool, date, purpose, and checks if you used one.
CHECK_PERFORMED = ""  # Describe one actual check, even if it found no error.

## Save and submit (5 min)
Complete your responses above, restart the runtime and run all. Download the `.ipynb` and generated summary JSON. In Colab the JSON is in the Files sidebar. Upload both to the course LMS assignment. Do not email patient data. A completion flag checks presence of responses, not scientific correctness. Paper fallback may be submitted as a legible scan with the same answers.

In [ ]:
required = [PREDICTION, CHECK_PERFORMED] + list(RESPONSES.values())
complete = all(isinstance(x,str) and x.strip() for x in required)
safe_id = "".join(c for c in COURSE_ID if c.isalnum() or c in "-_")[:40] or "anonymous"
report = {"week":3, "course_id":COURSE_ID,"seed":SEED,"python":sys.version.split()[0],
          "prediction":PREDICTION,"results":RESULTS,"responses":RESPONSES,
          "check_performed":CHECK_PERFORMED,"AI_disclosure":AI_DISCLOSURE,
          "response_fields_complete":complete}
output = Path(f"W03_{safe_id}_summary.json")
output.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Saved:",output)
print("Ready for review" if complete else "DRAFT: fill prediction, responses and check before submission")

## Paper / device-free route
Trace GCNN manually. The buggy fraction is 2/4, the known-base fraction is 2/2. Write three test cases.

## Optional extension
Optional: write a new test for lower-case letters and explain why it matters.

## Sources
- [S05] Python 3 tutorial: introduction and control flow. https://docs.python.org/3/tutorial/